# Preprocessing Papers
The plan is to pull out the text from pdf's, then split them into their abstracts, main content, and references for use in `project.ipynb`

In [7]:
# PyMuPDF to pull text from pdfs 

# extract texts from pdfs
import fitz # PyMuPDF
from pathlib import Path

def extract_text_from_pdf(pdf_path):
    text = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text.append(page.get_text('text'))
    return '\n'.join(text)

In [8]:
# Split data into sections

import re

def split_sections(text):
    # Normalize whitespace
    text = re.sub(r'\n+', '\n', text).strip()
    
    # Try to grab abstract
    abstract = ""
    body = text
    references = ""

    # Abstract extraction
    abs_match = re.search(r'(?is)(abstract)(.*?)(introduction|1\s)', text)
    if abs_match:
        abstract = abs_match.group(2).strip()
        body = text[abs_match.end():]

    # References extraction
    ref_match = re.search(r'(?is)(references|bibliography)(.*)', body)
    if ref_match:
        references = ref_match.group(2).strip()
        body = body[:ref_match.start()].strip()

    return abstract, body, references


In [10]:
# Save to JSON

import json

output_dir = Path("processed")
output_dir.mkdir(exist_ok=True)

def process_and_save(pdf_path):
    text = extract_text_from_pdf(pdf_path)
    abstract, body, references = split_sections(text)
    
    data = {
        "filename": pdf_path.name,
        "abstract": abstract,
        "body": body,
        "references": references
    }
    
    out_file = output_dir / f"{pdf_path.stem}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


In [12]:
# loop over papers to preprocess

papers_dir = Path("papers")
for pdf_file in papers_dir.glob("*.pdf"):
    process_and_save(pdf_file)
